<a href="https://colab.research.google.com/github/jivaniaadit/factor_xa_cheminformatics/blob/main/notebooks/week2_day4_test_train_splitting_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 47.8 MB/s eta 0:00:00


In [ ]:
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, rdMolDescriptors, rdFingerprintGenerator, AllChem, DataStructs
import pandas as pd
import numpy as np
from rdkit.Chem import MolStandardize
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df_clean = pd.read_csv('/content/drive/MyDrive/drugproj/factor_xa_clustered.csv')
print(df_clean.shape)
print(df_clean[['cluster_id', 'scaffold']].nunique())

Mounted at /content/drive
(3477, 16)
cluster_id     656
scaffold      1433
dtype: int64


In [ ]:
#Random split
train_idx, test_idx = train_test_split(
    df_clean.index, test_size=0.2, random_state=42
)
print(f"Random split: {len(train_idx)} train, {len(test_idx)} test")
train_scaffolds = set(df_clean.loc[train_idx, 'scaffold'])
test_scaffolds = set(df_clean.loc[test_idx, 'scaffold'])
shared = train_scaffolds & test_scaffolds

print(f"Scaffolds in both train and test: {len(shared)}")
print(f"As fraction of test scaffolds: {len(shared)/len(test_scaffolds):.1%}")

Random split: 2781 train, 696 test
Scaffolds in both train and test: 241
As fraction of test scaffolds: 55.5%


In [ ]:
#Proper scaffold split
def scaffold_split(df, test_frac=0.2, seed=42):
    rng = np.random.RandomState(seed)

    # group compound indices by scaffold
    scaffold_groups = df.groupby('scaffold').indices  # {scaffold: array of positions}
    scaffolds = list(scaffold_groups.keys())
    rng.shuffle(scaffolds)

    n_test_target = int(len(df) * test_frac)
    test_positions, train_positions = [], []

    for scaf in scaffolds:
        positions = scaffold_groups[scaf]
        if len(test_positions) < n_test_target:
            test_positions.extend(positions)
        else:
            train_positions.extend(positions)

    # convert positional indices to dataframe index labels
    train_labels = df.iloc[train_positions].index
    test_labels = df.iloc[test_positions].index
    return train_labels, test_labels

train_idx_s, test_idx_s = scaffold_split(df_clean, test_frac=0.2)
print(f"Scaffold split: {len(train_idx_s)} train, {len(test_idx_s)} test")

# verify zero leakage
train_scaf_s = set(df_clean.loc[train_idx_s, 'scaffold'])
test_scaf_s = set(df_clean.loc[test_idx_s, 'scaffold'])
print(f"Shared scaffolds: {len(train_scaf_s & test_scaf_s)}")

Scaffold split: 2764 train, 712 test
Shared scaffolds: 0


In [ ]:
#Comparing the two splits
print("Random split pKi:")
print(f"  train mean {df_clean.loc[train_idx, 'pKi'].mean():.2f}, "
      f"test mean {df_clean.loc[test_idx, 'pKi'].mean():.2f}")

print("Scaffold split pKi:")
print(f"  train mean {df_clean.loc[train_idx_s, 'pKi'].mean():.2f}, "
      f"test mean {df_clean.loc[test_idx_s, 'pKi'].mean():.2f}")
import pickle
splits = {
    'random': {'train': list(train_idx), 'test': list(test_idx)},
    'scaffold': {'train': list(train_idx_s), 'test': list(test_idx_s)},
}
with open('/content/drive/MyDrive/drugproj/splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

Random split pKi:
  train mean 7.11, test mean 7.16
Scaffold split pKi:
  train mean 7.13, test mean 7.06


In [ ]:
def cluster_split(df, test_frac=0.2, seed=42):
    rng = np.random.RandomState(seed)
    cluster_groups = df.groupby('cluster_id').indices
    clusters = list(cluster_groups.keys())
    rng.shuffle(clusters)

    n_test_target = int(len(df) * test_frac)
    test_positions, train_positions = [], []
    for c in clusters:
        positions = cluster_groups[c]
        if len(test_positions) < n_test_target:
            test_positions.extend(positions)
        else:
            train_positions.extend(positions)
    return df.iloc[train_positions].index, df.iloc[test_positions].index

train_idx_c, test_idx_c = cluster_split(df_clean, test_frac=0.2)
print(f"Cluster split: {len(train_idx_c)} train, {len(test_idx_c)} test")
splits['cluster'] = {'train': list(train_idx_c), 'test': list(test_idx_c)}
with open('/content/drive/MyDrive/drugproj/splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

Cluster split: 2775 train, 702 test
